# Minimum-Cost Food Supply Network (Linear Program)

A linear program that decides how much food to buy from each supplier and how to ship it
supplier → port → warehouse → camp at minimum total cost, while every camp receives its monthly
requirement of 11 nutrients.

- **Decision variables:** tons per month on every arc, for 14 commodities
- **Objective:** procurement + sea transport + land transport + handling at ports and warehouses
- **Constraints:** supplier, port and warehouse capacity; flow balance at every node; nutrition per camp

Built with `cvxpy`. Outputs are from the original run.

## 1. Data and decision variables

In [ ]:
import warnings

import cvxpy as cp
import pandas as pd

warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")

DATA_FILE = "Data OPT for DS.xlsx"  # course data, not included in this repository
sheets = ["Nodes", "Commodities", "Nutrients", "Nutritional values", "Procurement", "SeaTransport", "LandTransport"]
data = {sheet: pd.read_excel(DATA_FILE, sheet_name=sheet, engine="openpyxl") for sheet in sheets}

nodes = data["Nodes"]
commodities = data["Commodities"]
nutrients = data["Nutrients"]
nutritional_values = data["Nutritional values"]
procurement = data["Procurement"]
seatransport = data["SeaTransport"]
landtransport = data["LandTransport"]

# Sets ("LocationTYpe" is the column name used in the data file)
supplier = nodes[nodes["LocationTYpe"] == "Supplier"]["Location"].tolist()
port = nodes[nodes["LocationTYpe"] == "Port"]["Location"].tolist()
warehouse = nodes[nodes["LocationTYpe"] == "Warehouse"]["Location"].tolist()
beneficiarycamp = nodes[nodes["LocationTYpe"] == "Beneficiary Camp"]["Location"].tolist()
commodity = commodities["Commodity"].tolist()

# Parameters
procurement_capacity = procurement.set_index(["Supplier", "Commodity"])["Procurement capacity (ton/month)"].to_dict()
procurement_price = procurement.set_index(["Supplier", "Commodity"])["Procurement price ($/ton)"].to_dict()

hubs = nodes[nodes["LocationTYpe"].isin(["Port", "Warehouse"])].set_index("Location")
transport_capacity = hubs["Port capacity (mt/month)"].to_dict()
handling_price = hubs["Handling cost ($/ton)"].to_dict()
seatransport_price = seatransport.set_index(["Origin", "Destination", "Commodity"])["SeaTransport cost ($/ton)"].to_dict()
landtransport_price = landtransport.set_index(["Origin", "Destination"])["Landtransport cost ($/ton)"].to_dict()

nutrient_needs = nutrients.set_index("Nutrient")["Nutritional daily requirements (different units)"].to_dict()
beneficiary = nodes[nodes["Location"].isin(beneficiarycamp)].set_index("Location")["#Beneficiaries"].to_dict()

# Nutritional value per ton of each commodity (1 ton = 10,000 x 100 g)
nutritional_values_dict = {}
for _, row in nutritional_values.iterrows():
    comm = row["Commodity"].strip()
    nutr = row["Nutrient"].strip()
    nutritional_values_dict.setdefault(comm, {})[nutr] = float(row["Nutritional value (value/100 gram)!!"]) * 10000

# Decision variables (tons per month)
# x: bought from supplier s, y: shipped supplier -> port, z: port -> warehouse, q: warehouse -> camp
x = {(s, c): cp.Variable(nonneg=True) for (s, c) in procurement_capacity}
y = {(s, p, c): cp.Variable(nonneg=True) for (s, c) in x for p in port}
port_warehouse = [(p, w) for (p, w) in landtransport_price if p in port and w in warehouse]
warehouse_camp = [(w, b) for (w, b) in landtransport_price if w in warehouse and b in beneficiarycamp]
z = {(p, w, c): cp.Variable(nonneg=True) for (p, w) in port_warehouse for c in commodity}
q = {(w, b, c): cp.Variable(nonneg=True) for (w, b) in warehouse_camp for c in commodity}

## 2. Objective

In [ ]:
procurement_cost = sum(procurement_price[(s, c)] * x[(s, c)] for (s, c) in x)
seatransport_cost = sum(seatransport_price[(s, p, c)] * y[(s, p, c)] for (s, p, c) in y)
port_warehouse_cost = sum(landtransport_price[(p, w)] * z[(p, w, c)] for (p, w, c) in z)
warehouse_beneficiarycamp_cost = sum(landtransport_price[(w, b)] * q[(w, b, c)] for (w, b, c) in q)
port_handling_cost = sum(handling_price.get(p, 0) * y[(s, p, c)] for (s, p, c) in y)
warehouse_handling_cost = sum(handling_price.get(w, 0) * z[(p, w, c)] for (p, w, c) in z)

total_cost = (procurement_cost + seatransport_cost + port_warehouse_cost
              + warehouse_beneficiarycamp_cost + port_handling_cost + warehouse_handling_cost)
objective = cp.Minimize(total_cost)

## 3. Constraints

In [ ]:
constraints = []

# Nutrition: every camp receives its 30-day requirement of each of the 11 nutrients
for b in beneficiarycamp:
    for nutr, daily_req in nutrient_needs.items():
        req = beneficiary.get(b, 0) * daily_req * 30
        terms = [q[(w, b, c)] * nutritional_values_dict[c][nutr]
                 for w in warehouse for c in commodity
                 if (w, b, c) in q and nutr in nutritional_values_dict.get(c, {})]
        if terms:
            constraints.append(cp.sum(terms) >= req)

# Capacity: suppliers, ports and warehouses
for (s, c) in x:
    constraints.append(x[(s, c)] <= procurement_capacity[(s, c)])
for p in port:
    constraints.append(cp.sum([y[(s, p, c)] for s in supplier for c in commodity if (s, p, c) in y]) <= transport_capacity[p])
for w in warehouse:
    constraints.append(cp.sum([z[(p, w, c)] for p in port for c in commodity if (p, w, c) in z]) <= transport_capacity[w])

# Flow balance: flow in = flow out at suppliers, ports and warehouses
for (s, c) in x:
    constraints.append(x[(s, c)] == sum(y[(s, p, c)] for p in port if (s, p, c) in y))
for p in port:
    for c in commodity:
        inflow = sum(y[(s, p, c)] for s in supplier if (s, p, c) in y)
        outflow = sum(z[(p, w, c)] for w in warehouse if (p, w, c) in z)
        constraints.append(inflow == outflow)
for w in warehouse:
    for c in commodity:
        inflow = sum(z[(p, w, c)] for p in port if (p, w, c) in z)
        outflow = sum(q[(w, b, c)] for b in beneficiarycamp if (w, b, c) in q)
        constraints.append(inflow == outflow)

## 4. Solve

In [ ]:
problem = cp.Problem(objective, constraints)
problem.solve()

In [ ]:
# The default solver reports "optimal_inaccurate"; the checks at the end confirm the solution is feasible
print(f"Status: {problem.status}\n")
print(f"{'Total cost:':<20} {problem.value:.2f}")
print(f"{'procurement:':<20} {procurement_cost.value:.2f}")
print(f"{'seatransport:':<20} {seatransport_cost.value:.2f}")
print(f"{'port_warehouse:':<20} {port_warehouse_cost.value:.2f}")
print(f"{'warehouse_camp:':<20} {warehouse_beneficiarycamp_cost.value:.2f}")
print(f"{'port handling:':<20} {port_handling_cost.value:.2f}")
print(f"{'warehouse handling:':<20} {warehouse_handling_cost.value:.2f}")

x_values = {k: v.value for k, v in x.items()}
y_values = {k: v.value for k, v in y.items()}
z_values = {k: v.value for k, v in z.items()}
q_values = {k: v.value for k, v in q.items()}

Status: optimal_inaccurate

Optimal value:       36053317.24
procurement:         22547534.18
seatransport:        3743767.62
port_warehouse:      3510260.62
warehouse_camp:      3843781.59
port handling:       1525059.48
warehouse handling:  882913.75

## 5. Solution

In [ ]:
# Total amount bought per commodity, next to the two cheapest suppliers (a sanity check of the result)
food_total = {}
for (s, c), amount in x_values.items():
    food_total[c] = food_total.get(c, 0) + amount
sorted_food_total = sorted(food_total.items(), key=lambda item: item[1], reverse=True)

cheapest = {}
for c in commodities["Commodity"]:
    prices = sorted(((s, procurement_price[(s, c)]) for s in supplier if (s, c) in procurement_price),
                    key=lambda sp: sp[1])
    cheapest[c] = (prices + [("None", float("nan"))] * 2)[:2]

print(f"{'Commodity':<25}{'Total Amount (tons)':<25}{'Best Supplier 1':<25}{'Lowest Price 1':<25}{'Best Supplier 2':<25}{'Lowest Price 2':<25}\n")
for food, total in sorted_food_total:
    (s1, p1), (s2, p2) = cheapest[food]
    print(f"{food:<25}{total:<25.2f}{s1:<25}{p1:<25.2f}{s2:<25}{p2:<25.2f}")

Commodity                Total Amount (tons)      Best Supplier 1          Lowest Price 1           Best Supplier 2          Lowest Price 2           

WHEAT                    20773.87                 HUNGARY                  235.71                   UKRAINE                  237.24                   
SORGHUM/MILLET           15000.00                 INDIA                    218.99                   USA                      281.01                   
WHEAT FLOUR              15000.00                 TURKEY                   324.80                   USA                      537.68                   
CORN SOYA BLEND          10252.79                 TURKEY                   410.46                   SOUTH AFRICA             570.81                   
VEGETABLE OIL            2757.29                  INDONESIA                842.89                   BELGIUM                  1063.18                  
SPLIT PEAS               1000.00                  UKRAINE                  381.60            

### Purchases per supplier

In [ ]:
# Buy from Supplier
sorted_x_values = sorted(x_values.items(), key=lambda item: item[1], reverse=True)

print(f"{'Commodity':<25}{'Supplier':<25}{'Amount/tons':<15}\n")
for (s, c), amount in sorted_x_values:
    if amount > 1: 
        print(f"{c:<25}{s:<25}{amount:.2f}")
        

Commodity                Supplier                 Amount/tons    

WHEAT FLOUR              TURKEY                   15000.00
SORGHUM/MILLET           INDIA                    15000.00
WHEAT                    UKRAINE                  15000.00
WHEAT                    HUNGARY                  5773.87
CORN SOYA BLEND          TURKEY                   3000.00
CORN SOYA BLEND          SOUTH AFRICA             3000.00
CORN SOYA BLEND          ITALY                    3000.00
VEGETABLE OIL            INDONESIA                2000.00
CORN SOYA BLEND          BELGIUM                  1252.79
SPLIT PEAS               UKRAINE                  1000.00
VEGETABLE OIL            BELGIUM                  757.29
MAIZE                    SOUTH AFRICA             491.27
IODISED SALT             PAKISTAN                 202.17
IODISED SALT             NAMIBIA                  3.27

### Supplier → port

In [ ]:
# Supplier - Port
print(f"{'Supplier':<25}{'Port':<25}{'Commodity':<25}{'Amount/tons':<15}\n")
for (s, p, c), amount in y_values.items():
    if amount > 1:
        print(f"{s:<25}{p:<25}{c:<25}{amount:.2f}")
        

Supplier                 Port                     Commodity                Amount/tons    

BELGIUM                  DJIBOUTI (DJIBOUTI)      CORN SOYA BLEND          1252.79
BELGIUM                  DJIBOUTI (DJIBOUTI)      VEGETABLE OIL            757.29
HUNGARY                  BERBERA (SOMALIA)        WHEAT                    1024.53
HUNGARY                  DJIBOUTI (DJIBOUTI)      WHEAT                    4749.34
INDIA                    DJIBOUTI (DJIBOUTI)      SORGHUM/MILLET           15000.00
INDONESIA                DJIBOUTI (DJIBOUTI)      VEGETABLE OIL            2000.00
ITALY                    DJIBOUTI (DJIBOUTI)      CORN SOYA BLEND          3000.00
NAMIBIA                  BERBERA (SOMALIA)        IODISED SALT             3.27
PAKISTAN                 DJIBOUTI (DJIBOUTI)      IODISED SALT             202.17
SOUTH AFRICA             BERBERA (SOMALIA)        CORN SOYA BLEND          376.60
SOUTH AFRICA             DJIBOUTI (DJIBOUTI)      CORN SOYA BLEND          2623.40


### Port → warehouse

In [ ]:
# Port - Warehouse
print(f"{'Port':<25}{'Warehouse':<25}{'Commodity':<25}{'Amount/tons':<15}\n")
for (p, w, c), amount in z_values.items():
    if amount > 1:
        print(f"{p:<25}{w:<25}{c:<25}{amount:.2f}")
        

Port                     Warehouse                Commodity                Amount/tons    

DJIBOUTI (DJIBOUTI)      <JIJIGA>                 VEGETABLE OIL            51.61
DJIBOUTI (DJIBOUTI)      <MEKELE>                 SORGHUM/MILLET           130.65
DJIBOUTI (DJIBOUTI)      <MEKELE>                 WHEAT                    351.93
DJIBOUTI (DJIBOUTI)      <MEKELE>                 WHEAT FLOUR              359.58
DJIBOUTI (DJIBOUTI)      <MEKELE>                 IODISED SALT             5.83
DJIBOUTI (DJIBOUTI)      <MEKELE>                 VEGETABLE OIL            85.33
DJIBOUTI (DJIBOUTI)      <MEKELE>                 SPLIT PEAS               600.10
DJIBOUTI (DJIBOUTI)      <MEKELE>                 CORN SOYA BLEND          221.28
DJIBOUTI (DJIBOUTI)      <NAZARETH>               SORGHUM/MILLET           2401.51
DJIBOUTI (DJIBOUTI)      <NAZARETH>               WHEAT                    9310.31
DJIBOUTI (DJIBOUTI)      <NAZARETH>               WHEAT FLOUR              8415.25
DJIBOUT

### Warehouse → camp

In [ ]:
# Warehouse - Beneficiary Camp
print(f"{'Warehouse':<25}{'Beneficiarycamp':<25}{'Commodity':<25}{'Amount/tons':<15}\n")
for (w, b, c), amount in q_values.items():
    if amount > 1:
        print(f"{w:<25}{b:<25}{c:<25}{amount:.2f}")
        

Warehouse                Beneficiarycamp          Commodity                Amount/tons    

<DIRE DAWA>              OROMIYA                  SORGHUM/MILLET           5985.83
<DIRE DAWA>              OROMIYA                  WHEAT                    1801.11
<DIRE DAWA>              OROMIYA                  WHEAT FLOUR              1124.42
<DIRE DAWA>              OROMIYA                  IODISED SALT             33.82
<DIRE DAWA>              OROMIYA                  VEGETABLE OIL            393.23
<DIRE DAWA>              OROMIYA                  SPLIT PEAS               5.38
<DIRE DAWA>              OROMIYA                  CORN SOYA BLEND          1791.70
<DIRE DAWA>              SOMALI                   SORGHUM/MILLET           1455.97
<DIRE DAWA>              SOMALI                   WHEAT                    4846.23
<DIRE DAWA>              SOMALI                   WHEAT FLOUR              5100.72
<DIRE DAWA>              SOMALI                   IODISED SALT             47.40
<DI

## 6. Checks

In [ ]:
# Check: nutrition supplied to the SOMALI camp versus its 30-day requirement
b = "SOMALI"
rows = []
for nutr, daily_req in nutrient_needs.items():
    demand = beneficiary.get(b, 0) * daily_req * 30
    supply = sum(q_values[(w, b, c)] * nutritional_values_dict[c][nutr]
                 for w in warehouse for c in commodity
                 if (w, b, c) in q_values and nutr in nutritional_values_dict.get(c, {}))
    percentage = (supply / demand) * 100 if demand > 0 else 0
    rows.append([nutr, supply, demand, f"{percentage:.2f}%", "Y" if supply >= demand else "N"])

nutrition_status_df = pd.DataFrame(rows, columns=["Nutrient", "Supply", "Demand", "Supply/Demand (%)", "Status"])
print(nutrition_status_df)

      Nutrient        Supply        Demand Supply/Demand (%) Status
0       Energy  5.265144e+10  5.265144e+10           100.00%      Y
1      Protein  1.583789e+09  1.316286e+09           120.32%      Y
2          Fat  1.002885e+09  1.002885e+09           100.00%      Y
3      Calcium  1.128246e+10  1.128245e+10           100.00%      Y
4         Iron  5.704044e+08  5.515865e+08           103.41%      Y
5       Iodine  3.760820e+09  3.760817e+09           100.00%      Y
6       Vit. A  2.476405e+10  1.253606e+10           197.54%      Y
7     Thiamine  5.880313e+07  2.256490e+07           260.60%      Y
8   Riboflavin  3.510096e+07  3.510096e+07           100.00%      Y
9       Niacin  9.594546e+08  3.474995e+08           276.10%      Y
10      Vit. C  1.929694e+09  7.020192e+08           274.88%      Y

In [ ]:
# Check: flow in = flow out at the port of Djibouti
djibouti = "DJIBOUTI (DJIBOUTI)"
import_in = sum(v for (s, p, c), v in y_values.items() if p == djibouti)
export_out = sum(v for (p, w, c), v in z_values.items() if p == djibouti)
print(f"DJIBOUTI in: {import_in:.2f}")
print(f"DJIBOUTI out: {export_out:.2f} ")

DJIBOUTI in: 64076.27
DJIBOUTI out: 64076.27 

In [ ]:
# Check: corn soya blend delivered to all camps = corn soya blend procured
total_corn_soya_blend = sum(v for (w, b, c), v in q_values.items() if c == "CORN SOYA BLEND")
procurement_csb = sum(v for (s, c), v in x_values.items() if c == "CORN SOYA BLEND")
print(f"Total CORN SOYA BLEND received by all beneficiary camps: {total_corn_soya_blend:.2f}")
print(f"Total Procurement of CORN SOYA BLEND: {procurement_csb:.2f}")

Total CORN SOYA BLEND received by all beneficiary camps: 10252.79
Total Procurement of CORN SOYA BLEND: 10252.79